In [1]:
# =========================================
# Import required libraries
# =========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
# =========================================
# load Dataset
# =========================================

processed_data_path = "../data/processed"

df = pd.read_csv(f"{processed_data_path}/municipal_data_2024_ml.csv")

print(df.head())
print(df.info())

   Year        Municipality  Population  Mortality_per_1000  \
0  2024            Abrantes       33770                16.5   
1  2024              Agueda       47727                11.4   
2  2024     Aguiar Da Beira        5326                23.5   
3  2024           Alandroal        4973                21.1   
4  2024  Albergaria-A-Velha       26171                10.2   

   Infant_mortality_per_1000  Births_per_1000  Unemployed_total  \
0                        0.0              7.0            1328.0   
1                        0.0              7.3            1157.0   
2                        0.0              5.1              82.0   
3                        0.0              5.8              81.0   
4                        5.7              6.8             516.0   

   Average_income  Pharmacies_total  Total_crimes  Marriages_per_1000  \
0          1289.0                15           840                 2.6   
1          1280.2                14          1420                 1.4   

In [3]:
# =========================================
# Derived and Normalized Variables
# =========================================

# Population balance: birth minus deaths per 1000 inhabitants
df["Population_balance"] = df["Births_per_1000"] - df["Mortality_per_1000"]

# Function to add a normalized rate column 
def add_rate(df, numerator, denominator, multiplier, new_col):
    df[new_col] = df[numerator] / df[denominator] * multiplier

# Create normalized indicators 
add_rate(df, "Total_crimes", "Population", 1000, "Crimes_per_1000")
add_rate(df, "Unemployed_total", "Population", 100, "Unemployment_rate")
add_rate(df, "Pharmacies_total", "Population", 10000, "Pharmacies_per_10000")

# Drop raw totals 
df = df.drop(columns=[
    "Total_crimes",
    "Unemployed_total",
    "Pharmacies_total"
])

print(df.head())
print(df.info())

   Year        Municipality  Population  Mortality_per_1000  \
0  2024            Abrantes       33770                16.5   
1  2024              Agueda       47727                11.4   
2  2024     Aguiar Da Beira        5326                23.5   
3  2024           Alandroal        4973                21.1   
4  2024  Albergaria-A-Velha       26171                10.2   

   Infant_mortality_per_1000  Births_per_1000  Average_income  \
0                        0.0              7.0          1289.0   
1                        0.0              7.3          1280.2   
2                        0.0              5.1          1016.9   
3                        0.0              5.8          1171.4   
4                        5.7              6.8          1349.1   

   Marriages_per_1000  Divorces_per_1000  Median_price_sqm  \
0                 2.6                1.4             717.0   
1                 1.4                1.8             960.0   
2                 3.2                1.3    

In [4]:
# =========================================
# Define Target (y)
# =========================================

# Drop missing values from target
df = df.dropna(subset=["Median_price_sqm"])
y = df["Median_price_sqm"]

In [5]:
# =========================================
# Define Features (X)
# =========================================

X = df.drop(columns=["Year", "Municipality", "Median_price_sqm"]) # while droping the unecessary columns + target

print(X.head())

   Population  Mortality_per_1000  Infant_mortality_per_1000  Births_per_1000  \
0       33770                16.5                        0.0              7.0   
1       47727                11.4                        0.0              7.3   
2        5326                23.5                        0.0              5.1   
3        4973                21.1                        0.0              5.8   
4       26171                10.2                        5.7              6.8   

   Average_income  Marriages_per_1000  Divorces_per_1000  Population_balance  \
0          1289.0                 2.6                1.4                -9.5   
1          1280.2                 1.4                1.8                -4.1   
2          1016.9                 3.2                1.3               -18.4   
3          1171.4                 4.8                1.8               -15.3   
4          1349.1                 4.0                1.6                -3.4   

   Crimes_per_1000  Unemployment

In [6]:
# =========================================
# Confirm Shapes
# =========================================

print(X.shape)
print(y.shape)

(300, 11)
(300,)


In [7]:
# =========================================
# Train/Test Split
# =========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y,
    random_state=42
)

# Sanity check (verify sizes): Expected train = 75%, test = 25%
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(225, 11)
(75, 11)
(225,)
(75,)


In [8]:
# =========================================
# Create Pipeline
# =========================================

lr_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")), # fill NaNs with column mean
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

# Fit the model
lr_pipeline.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler()),
                ('model', LinearRegression())])

In [9]:
# =========================================
# Making Predictions
# =========================================

y_pred = lr_pipeline.predict(X_test)

In [10]:
# =========================================
# Comparison Real vs. Predicted Values
# =========================================

comparison = pd.DataFrame({
    "Real": y_test,
    "Predicted": y_pred
})

comparison.head()

,Real,Predicted
209,2757.0,3086.987915
274,1715.0,1630.043280
157,404.0,541.783615
9,2073.0,1848.308206
240,772.0,368.757623


In [11]:
# =========================================
# Model Evaluation
# =========================================

print(f"Training set score: {lr_pipeline.score(X_train, y_train):.2f}")
print(f"Test set score: {lr_pipeline.score(X_test, y_test):.2f}")

Training set score: 0.67
Test set score: 0.78


In [12]:
# =========================================
# Cross-validation
# =========================================

from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    lr_pipeline,
    X,
    y,
    cv=5,
    scoring="r2"
)

print(f"Scores: {scores}")
print(f"Average score: {scores.mean():.2f}")

Scores: [0.72120725 0.65894616 0.74670478 0.52568743 0.60292863]
Average score: 0.65


In [13]:
# =========================================
# Linear Regression Coefficients
# =========================================

lr_model = lr_pipeline.named_steps["model"]

coefficients = lr_model.coef_

lr_importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": coefficients
}).sort_values(by="Coefficient", key=abs, ascending=False)

lr_importance

,Feature,Coefficient
0,Population,251.853465
8,Crimes_per_1000,211.621776
5,Marriages_per_1000,115.314687
7,Population_balance,106.950993
1,Mortality_per_1000,-103.931084
10,Pharmacies_per_10000,-79.516090
3,Births_per_1000,78.068222
9,Unemployment_rate,-65.396878
4,Average_income,45.101952
6,Divorces_per_1000,-33.870321


In [14]:
# =========================================
# Ridge Regression
# =========================================

ridge_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

ridge_pipeline.fit(X_train, y_train)

print(f"Ridge training set score: {ridge_pipeline.score(X_train, y_train):.2f}")
print(f"Ridge test set score: {ridge_pipeline.score(X_test, y_test):.2f}")

Ridge training set score: 0.67
Ridge test set score: 0.79


In [15]:
# =========================================
# Ridge Regression Coefficients
# =========================================

coef = ridge_pipeline.named_steps["ridge"].coef_
features = X.columns

for f, c in sorted(zip(features, coef), key=lambda x: abs(x[1]), reverse=True):
    print(f"{f:25} {c:.3f}")

Population                250.171
Crimes_per_1000           210.213
Marriages_per_1000        114.722
Population_balance        106.805
Mortality_per_1000        -103.372
Pharmacies_per_10000      -79.415
Births_per_1000           79.089
Unemployment_rate         -64.995
Average_income            45.700
Divorces_per_1000         -33.412
Infant_mortality_per_1000 -11.395


In [16]:
# =========================================
# Ridge Regression Cross-validation
# =========================================

ridge_scores = cross_val_score(
    ridge_pipeline,
    X,
    y,
    cv=5,
    scoring="r2"
)

print(f"Scores: {ridge_scores}")
print(f"Average score: {ridge_scores.mean():.2f}")

Scores: [0.72177897 0.65849685 0.74632309 0.52764151 0.60371223]
Average score: 0.65


In [17]:
# =========================================
# Random Forest Regressor
# =========================================

rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)

print(f"Random Forest training set score: {rf_pipeline.score(X_train, y_train):.2f}")
print(f"Random Forest test set score: {rf_pipeline.score(X_test, y_test):.2f}")

Random Forest training set score: 0.95
Random Forest test set score: 0.74


In [18]:
# =========================================
# Random Forest Regressor Cross-validation
# =========================================

rf_scores = cross_val_score(
    rf_pipeline,
    X,
    y,
    cv=5,
    scoring="r2"
)

print(f"Scores: {rf_scores}")
print(f"Average score: {rf_scores.mean():.2f}")

Scores: [0.65713362 0.5974891  0.70788488 0.51660706 0.64991597]
Average score: 0.63


In [19]:
# =========================================
# Feature Importance
# =========================================

rf_model = rf_pipeline.named_steps["model"]

importances = rf_model.feature_importances_

rf_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print(rf_importance)

                      Feature  Importance
7          Population_balance    0.432293
0                  Population    0.137732
8             Crimes_per_1000    0.104163
1          Mortality_per_1000    0.072658
4              Average_income    0.052437
5          Marriages_per_1000    0.052094
3             Births_per_1000    0.051184
2   Infant_mortality_per_1000    0.027644
10       Pharmacies_per_10000    0.025743
9           Unemployment_rate    0.025703
6           Divorces_per_1000    0.018350


In [20]:
# =========================================
# Model Comparison Table
# =========================================

results = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge", "Random Forest"],
    "Train_R2": [
        lr_pipeline.score(X_train, y_train),
        ridge_pipeline.score(X_train, y_train),
        rf_pipeline.score(X_train, y_train)
    ],
    "Test_R2": [
        lr_pipeline.score(X_test, y_test),
        ridge_pipeline.score(X_test, y_test),
        rf_pipeline.score(X_test, y_test)
    ],
    "CV_Mean_R2": [
        scores.mean(),
        ridge_scores.mean(),
        rf_scores.mean()
    ]
})

results.round(3)

,Model,Train_R2,Test_R2,CV_Mean_R2
0,Linear Regression,0.667,0.784,0.651
1,Ridge,0.667,0.786,0.652
2,Random Forest,0.948,0.740,0.626


In [21]:
# =========================================
# No Population
# =========================================

X_no_pop = X.drop(columns=["Population"])

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_no_pop, 
    y, 
    random_state=42
)

# Linear Regression 
lr_pipeline.fit(X_train_np, y_train_np)

# Ridge Regression 
ridge_pipeline.fit(X_train_np, y_train_np) 

# Random Forest 
rf_pipeline.fit(X_train_np, y_train_np) 

results_no_pop = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge", "Random Forest"],
    "Train_R2": [
        lr_pipeline.score(X_train_np, y_train_np),
        ridge_pipeline.score(X_train_np, y_train_np),
        rf_pipeline.score(X_train_np, y_train_np)
    ],
    "Test_R2": [
        lr_pipeline.score(X_test_np, y_test_np),
        ridge_pipeline.score(X_test_np, y_test_np),
        rf_pipeline.score(X_test_np, y_test_np)
    ]
})

results_no_pop.round(3)

,Model,Train_R2,Test_R2
0,Linear Regression,0.589,0.690
1,Ridge,0.589,0.690
2,Random Forest,0.939,0.675
